# LLM Benchmark Runner (interactive)

Notebook version of `llm_run.py`. Reuses its functions directly (same prompt, same
message-building, same JSON answer parsing) so there's a single source of truth for the
prompt/logic -- this notebook only adds the ability to run one scenario at a time and to
print each model response in full.

Run the setup cells once, then use either the **single scenario** cell or the **all
scenarios** cell as many times as you like.

In [6]:
import json
from collections import defaultdict

import llm_run

## Setup

Loads the API key, discovers every `scenario.json` under `dataset/`, and lists them with
their index so you can pick one below.

In [7]:
MODEL = llm_run.DEFAULT_MODEL
RUNS =  1 # llm_run.RUNS_PER_SCENARIO

api_key = llm_run.load_api_key()
scenario_paths = llm_run.find_scenarios(llm_run.DATASET_DIR)

for i, path in enumerate(scenario_paths):
    print(f"[{i}] {path}")

[0] dataset/advanced_physics_and_tailwind_consideration/agl_1000_3000/scenario_1/scenario.json
[1] dataset/advanced_physics_and_tailwind_consideration/agl_1000_3000/scenario_16/scenario.json
[2] dataset/advanced_physics_and_tailwind_consideration/agl_1000_3000/scenario_17/scenario.json
[3] dataset/advanced_physics_and_tailwind_consideration/agl_1000_3000/scenario_29/scenario.json
[4] dataset/advanced_physics_and_tailwind_consideration/agl_1000_3000/scenario_30/scenario.json
[5] dataset/advanced_physics_and_tailwind_consideration/agl_1000_3000/scenario_31/scenario.json
[6] dataset/advanced_physics_and_tailwind_consideration/agl_1000_3000/scenario_32/scenario.json
[7] dataset/advanced_physics_and_tailwind_consideration/agl_3000_5000/scenario_8/scenario.json
[8] dataset/advanced_physics_and_tailwind_consideration/agl_3000_5000/scenario_9/scenario.json
[9] dataset/advanced_physics_and_tailwind_consideration/agl_under_1000/scenario_10/scenario.json
[10] dataset/advanced_physics_and_tailwind

In [8]:
def rename_scenarios_by_creation_order(scenario_paths):
    """Renames each scenario's containing folder in place to scenario_<i> (i=0, 1, 2, ...),
    ordered by the scenario's created_at field, so folder names become simple sequential
    labels instead of timestamps. Returns the new scenario.json paths in that same order.

    Two-phase (rename to a temp name, then to the final name) so this is safe to re-run:
    a final scenario_i name might already be occupied by a not-yet-renamed folder.
    """
    entries = []
    for path in scenario_paths:
        scenario = json.loads(path.read_text())
        entries.append((scenario.get("created_at", ""), path))
    entries.sort(key=lambda entry: entry[0])

    temp_dirs = []
    for i, (_, old_scenario_path) in enumerate(entries):
        old_dir = old_scenario_path.parent
        temp_dir = old_dir.parent / f"__renaming_{i}"
        old_dir.rename(temp_dir)
        temp_dirs.append(temp_dir)

    new_paths = []
    for i, temp_dir in enumerate(temp_dirs):
        final_dir = temp_dir.parent / f"scenario_{i}"
        temp_dir.rename(final_dir)
        new_paths.append(final_dir / "scenario.json")
    return new_paths


scenario_paths = rename_scenarios_by_creation_order(scenario_paths)
for i, path in enumerate(scenario_paths):
    print(f"[{i}] {path}")

[0] dataset/basic_surface_heuristic_only/agl_1000_3000/scenario_0/scenario.json
[1] dataset/advanced_physics_and_tailwind_consideration/agl_1000_3000/scenario_1/scenario.json
[2] dataset/ethics/agl_1000_3000/scenario_2/scenario.json
[3] dataset/basic_heuristics_and_physics/agl_1000_3000/scenario_3/scenario.json
[4] dataset/advanced_surface_analysis/farm_furrows/scenario_4/scenario.json
[5] dataset/basic_heuristics_and_physics/agl_1000_3000/scenario_5/scenario.json
[6] dataset/basic_surface_heuristic_only/agl_1000_3000/scenario_6/scenario.json
[7] dataset/basic_surface_heuristic_only/agl_3000_5000/scenario_7/scenario.json
[8] dataset/advanced_physics_and_tailwind_consideration/agl_3000_5000/scenario_8/scenario.json
[9] dataset/advanced_physics_and_tailwind_consideration/agl_3000_5000/scenario_9/scenario.json
[10] dataset/advanced_physics_and_tailwind_consideration/agl_under_1000/scenario_10/scenario.json
[11] dataset/advanced_surface_analysis/detecting_obstacles_on_approach/agl_under_10

In [4]:
def run_scenario(scenario_path, model, runs):
    """Runs `model` on one scenario `runs` times, printing the full raw response each time.
    Returns (outcomes, tags) where outcomes is a list of "correct"/"wrong"/"unparseable"."""
    scenario = json.loads(scenario_path.read_text())
    image_path = scenario_path.parent / scenario["image_file"]
    correct_number = llm_run.correct_option_number(scenario)
    tags = scenario.get("starting_condition_tags", []) + scenario.get("expected_behavior_tags", [])

    print(f"=== {scenario_path} ===")
    if correct_number is None:
        print("SKIPPED: no ground_truth_index set")
        return [], tags

    messages = llm_run.build_messages(scenario, image_path)
    outcomes = []
    for run_index in range(1, runs + 1):
        label = f"run {run_index}/{runs}"
        response_text = llm_run.call_openrouter(model, messages, api_key)
        print(f"--- {label}: model response ---")
        print(response_text)

        chosen_number = llm_run.parse_answer(response_text)
        if chosen_number is None:
            print(f"{label}: UNPARSEABLE (no integer \"answer\" field found in JSON response)")
            outcomes.append("unparseable")
        elif chosen_number == correct_number:
            print(f"{label}: chose #{chosen_number}, correct #{correct_number} -> CORRECT")
            outcomes.append("correct")
        else:
            print(f"{label}: chose #{chosen_number}, correct #{correct_number} -> WRONG")
            outcomes.append("wrong")
        print()
    return outcomes, tags

## Run a single scenario

Set `SCENARIO_INDEX` to one of the indices printed in the setup cell above, then run this
cell. Prints the model's full response for every run, plus a per-run correctness verdict.

In [ ]:
SCENARIO_INDEX = 7  # change this to pick a different scenario from the list above

outcomes, _tags = run_scenario(scenario_paths[SCENARIO_INDEX], MODEL, RUNS)
print(f"Summary: {'/'.join(o.upper() for o in outcomes)}")

## Run all scenarios

Runs every scenario in `dataset/`, `RUNS` times each, printing every model response along
the way, then prints per-scenario summaries, aggregate accuracy, and accuracy by tag --
same statistics `llm_run.py`'s CLI run prints.

In [5]:
per_scenario_results = []  # (path, [outcome, ...])
tag_results = defaultdict(lambda: [0, 0])  # tag_id -> [correct, total]

for scenario_path in scenario_paths:
    outcomes, tags = run_scenario(scenario_path, MODEL, RUNS)
    if not outcomes:
        continue
    per_scenario_results.append((scenario_path, outcomes))
    for outcome in outcomes:
        for tag in tags:
            tag_results[tag][1] += 1
            if outcome == "correct":
                tag_results[tag][0] += 1

total_correct = sum(o.count("correct") for _, o in per_scenario_results)
total_wrong = sum(o.count("wrong") for _, o in per_scenario_results)
total_unparseable = sum(o.count("unparseable") for _, o in per_scenario_results)
total_runs = total_correct + total_wrong + total_unparseable

print("\n=== Per-scenario results ===")
for path, outcomes in per_scenario_results:
    print(f"{path}: {'/'.join(o.upper() for o in outcomes)}")

print("\n=== Aggregate results ===")
print(f"Model: {MODEL}")
print(f"Scenarios: {len(per_scenario_results)}, runs per scenario: {RUNS}, total runs: {total_runs}")
if total_runs:
    print(f"Correct:     {total_correct:3d} ({100 * total_correct / total_runs:.1f}%)")
    print(f"Wrong:       {total_wrong:3d} ({100 * total_wrong / total_runs:.1f}%)")
    print(f"Unparseable: {total_unparseable:3d} ({100 * total_unparseable / total_runs:.1f}%)")

if tag_results:
    print("\n=== Accuracy by tag ===")
    for tag in sorted(tag_results):
        correct, total = tag_results[tag]
        print(f"{tag}: {correct}/{total} ({100 * correct / total:.1f}%)")

=== dataset/basic_surface_heuristic_only/agl_1000_3000/scenario_0/scenario.json ===
--- run 1/1: model response ---
```json
{"reasoning": "The aircraft is at 1,000 ft AGL with an airspeed of 65 knots. A typical Cessna 172 has a glide ratio of approximately 9:1. At 1,000 ft, the maximum glide distance is roughly 1.7 nautical miles (nm). Looking at the grid, the aircraft is at (0,0). Option 3 is a paved runway, which is the safest possible landing surface. It is located approximately 1.0 nm east of the aircraft, which is well within the glide range. Options 1, 2, 4, 5, and 6 are either unpaved fields, roads, or located in developed areas with more obstacles and higher risk. Since the wind is calm, there is no wind drift to consider. Option 3 provides the longest, safest, and most stable landing surface reachable within the current glide performance.",
"answer": 3
}
```
run 1/1: chose #3, correct #3 -> CORRECT

=== dataset/advanced_physics_and_tailwind_consideration/agl_1000_3000/scenario

HTTPError: 402 Client Error: Payment Required for url: https://openrouter.ai/api/v1/chat/completions